In [ ]:
import glob
import os
import shutil

REPO = "/kaggle/working/small-model-alignment"
found = glob.glob("/kaggle/input/**/pa2-code/**/alignment_core.py", recursive=True)
print(found)
if not os.path.exists(REPO):
    shutil.copytree(os.path.dirname(found[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
print(os.listdir(REPO))

In [ ]:
!nvidia-smi
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
assert _exit_code == 0

In [ ]:
import datasets
import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())
print("model proxy available", "MODEL_PROXY_URL" in os.environ)

In [ ]:
sft_config = glob.glob("/kaggle/input/**/outputs/sft/best/config.json", recursive=True)
print(sft_config)
SFT_PATH = os.path.dirname(sft_config[0])
SFT_GENERATIONS = os.path.dirname(glob.glob("/kaggle/input/**/outputs/generations/sft_alpaca.json", recursive=True)[0])
ALPACA_EVAL = glob.glob("/kaggle/input/**/eval_sets/alpaca_eval_300_seed42.json", recursive=True)[0]
print(SFT_PATH, SFT_GENERATIONS, ALPACA_EVAL)

In [ ]:
!python scripts/train_dpo.py --dataset hh --sft_path $SFT_PATH --output_dir outputs/dpo_hh --train_size 15000 --val_size 1000 --eval_every 100
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/dpo_hh/metrics.jsonl --output_dir outputs/plots --run_name dpo_hh --kind dpo
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sanity_dpo_hh_hh.json --generator qwen2.5-0.5b-dpo-hh --dataset hh_rlhf_style_eval --limit 3 --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/dpo_hh_hh.json --generator qwen2.5-0.5b-dpo-hh --dataset hh_rlhf_style_eval --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/build_pairs.py --baseline_path $SFT_GENERATIONS/sft_hh.json --candidate_path outputs/generations/dpo_hh_hh.json --output_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/dpo_hh_run_outputs.zip outputs -x 'outputs/*/best/*' 'outputs/*/final/*'
assert _exit_code == 0